In [1]:
import numpy as np
import pandas as pd
import datetime

In [2]:
import importlib
import ASFScroller
importlib.reload(ASFScroller)

scroll_new = False

if (scroll_new):
    scroller = ASFScroller.SICEScroller()
    scroller.search_geo(c_min=0.2, c_max=0.8, v_min=0, v_max=0.0005, ll_sample_len=(0.5,1))
    scroller.save_cache()
else:
    scroller = ASFScroller.SICEScroller.from_cache('20070101-20120101-Scroller.pickle')

In [3]:
scroller.results

{'2008-06-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x7616bc16c680>,
 '2009-07-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x76167685bc20>,
 '2007-05-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x76166734da30>,
 '2007-06-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x761666e95a00>,
 '2010-07-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x761666f1ab70>,
 '2008-05-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x761666f59100>,
 '2009-06-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x761666da8c80>,
 '2009-05-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x761666bf5d90>,
 '2010-06-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x7616669f3590>,
 '2009-08-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x761666a2b080>,
 '2009-04-01': [<asf_search.Products.ALOSProduct.ALOSProduct at 0x76166693f680>,
  <asf_search.Products.ALOSProduct.ALOSProduct at 0x7616667805f0>]}

In [4]:
# SBAS SEARCH

prodl = np.array([(key, val, val.properties['sceneName']) for (key, vals) in scroller.results.items() for val in vals])
prodl = prodl[np.unique(prodl[:,2], return_index=True)[1]] # unique scenes

sbasl = []
for el in prodl:
    try: sbasl.append(el[1].stack())
    except: sbasl.append([])


In [7]:
#len(prodb[prodb['tsep'] < datetime.timedelta(days=50)].sort_values(by='tosp'))

In [34]:
prodb = pd.DataFrame(prodl, columns=['date','result','scene_id'])
prodb['date'] = prodb['result'].apply(lambda x: datetime.datetime.strptime(x.properties['startTime'], "%Y-%m-%dT%H:%M:%SZ"))
prodb['sbas_result'] = sbasl
for i in prodb.index:
    match = prodb.loc[i,'scene_id']
    tmp = [el for el in prodb.loc[i,'sbas_result'] if el.properties['sceneName'] != match]
    if (len(tmp) == 0):
        prodb.loc[i, 'sbas_result'] = None
        continue
    tmp1 = np.array([datetime.datetime.strptime(el.properties['startTime'],"%Y-%m-%dT%H:%M:%SZ") for el in tmp])
    prodb.loc[i,'sbas_result'] = tmp[np.argmin(abs(tmp1 - np.array([prodb.loc[i,'date']])))]
prodb = prodb.dropna(axis=0)
prodb['pair_date'] = prodb['sbas_result'].apply(lambda x: datetime.datetime.strptime(x.properties['startTime'], "%Y-%m-%dT%H:%M:%SZ"))
prodb['scene_id'] += ", " + prodb['sbas_result'].apply(lambda x: x.properties['sceneName'])

target_month = 2

prodb['tsep'] = abs(prodb['pair_date'] - prodb['date'])
prodb['tosp'] = prodb['date'].apply(lambda x: abs(x.month + (x.day/30) - target_month))
tmp = prodb[prodb['tsep'] < datetime.timedelta(days=50)].sort_values(by='tosp')

tmp1 = ""
for scene_id in tmp['scene_id']:
    tmp1 += ", " + scene_id
print(tmp1[2:])

ALPSRP067391560, ALPSRP060681560, ALPSRP067391570, ALPSRP060681570, ALPSRP067521420, ALPSRP060811420, ALPSRP068281250, ALPSRP061571250, ALPSRP068301420, ALPSRP061591420, ALPSRP068301430, ALPSRP061591430, ALPSRP068411540, ALPSRP061701540, ALPSRP068411600, ALPSRP061701600, ALPSRP068601390, ALPSRP061891390, ALPSRP069141540, ALPSRP062431540, ALPSRP069141560, ALPSRP062431560, ALPSRP069901400, ALPSRP063191400, ALPSRP069901410, ALPSRP063191410, ALPSRP070161530, ALPSRP063451530, ALPSRP123581400, ALPSRP116871400, ALPSRP123581410, ALPSRP116871410, ALPSRP070301320, ALPSRP063591320, ALPSRP070301310, ALPSRP063591310, ALPSRP071181420, ALPSRP064471420, ALPSRP071621550, ALPSRP064911550, ALPSRP071751420, ALPSRP065041420, ALPSRP071621570, ALPSRP064911570, ALPSRP071621560, ALPSRP064911560, ALPSRP072281560, ALPSRP065571560, ALPSRP125921430, ALPSRP119211430, ALPSRP072531420, ALPSRP065821420, ALPSRP072641600, ALPSRP065931600, ALPSRP072641610, ALPSRP065931610, ALPSRP233871330, ALPSRP227161330, ALPSRP12840143

In [35]:
print(', '.join([el+"-L1.1" for el in tmp1[2:].split(', ')]))

ALPSRP067391560-L1.1, ALPSRP060681560-L1.1, ALPSRP067391570-L1.1, ALPSRP060681570-L1.1, ALPSRP067521420-L1.1, ALPSRP060811420-L1.1, ALPSRP068281250-L1.1, ALPSRP061571250-L1.1, ALPSRP068301420-L1.1, ALPSRP061591420-L1.1, ALPSRP068301430-L1.1, ALPSRP061591430-L1.1, ALPSRP068411540-L1.1, ALPSRP061701540-L1.1, ALPSRP068411600-L1.1, ALPSRP061701600-L1.1, ALPSRP068601390-L1.1, ALPSRP061891390-L1.1, ALPSRP069141540-L1.1, ALPSRP062431540-L1.1, ALPSRP069141560-L1.1, ALPSRP062431560-L1.1, ALPSRP069901400-L1.1, ALPSRP063191400-L1.1, ALPSRP069901410-L1.1, ALPSRP063191410-L1.1, ALPSRP070161530-L1.1, ALPSRP063451530-L1.1, ALPSRP123581400-L1.1, ALPSRP116871400-L1.1, ALPSRP123581410-L1.1, ALPSRP116871410-L1.1, ALPSRP070301320-L1.1, ALPSRP063591320-L1.1, ALPSRP070301310-L1.1, ALPSRP063591310-L1.1, ALPSRP071181420-L1.1, ALPSRP064471420-L1.1, ALPSRP071621550-L1.1, ALPSRP064911550-L1.1, ALPSRP071751420-L1.1, ALPSRP065041420-L1.1, ALPSRP071621570-L1.1, ALPSRP064911570-L1.1, ALPSRP071621560-L1.1, ALPSRP0649

In [38]:
prodb

,date,result,scene_id,sbas_result,pair_date,tsep,tosp
0,2007-05-01 01:35:27,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP067391560, ALPSRP060681560","{\n ""geometry"": {\n ""coordinates"": [\n ...",2007-03-16 01:35:27,46 days 00:00:00,3.033333
1,2007-05-01 01:35:36,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP067391570, ALPSRP060681570","{\n ""geometry"": {\n ""coordinates"": [\n ...",2007-03-16 01:35:36,46 days 00:00:00,3.033333
2,2007-05-01 22:56:53,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP067521420, ALPSRP060811420","{\n ""geometry"": {\n ""coordinates"": [\n ...",2007-03-16 22:56:53,46 days 00:00:00,3.033333
3,2007-05-05 00:59:29,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP067971440, ALPSRP168621440","{\n ""geometry"": {\n ""coordinates"": [\n ...",2009-03-25 00:59:56,690 days 00:00:27,3.166667
4,2007-05-05 01:00:18,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP067971500, ALPSRP168621500","{\n ""geometry"": {\n ""coordinates"": [\n ...",2009-03-25 01:00:45,690 days 00:00:27,3.166667
...,...,...,...,...,...,...,...
101,2009-08-07 05:34:16,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP188341500, ALPSRP201761500","{\n ""geometry"": {\n ""coordinates"": [\n ...",2009-11-07 05:34:31,92 days 00:00:15,6.233333
103,2009-08-11 05:00:46,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP188921560, ALPSRP202341560","{\n ""geometry"": {\n ""coordinates"": [\n ...",2009-11-11 05:01:00,92 days 00:00:14,6.366667
104,2010-06-15 08:35:32,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP233871330, ALPSRP227161330","{\n ""geometry"": {\n ""coordinates"": [\n ...",2010-04-30 08:35:59,45 days 23:59:33,4.500000
105,2010-06-23 07:28:09,"{\n ""geometry"": {\n ""coordinates"": [\n ...","ALPSRP235031430, ALPSRP228321430","{\n ""geometry"": {\n ""coordinates"": [\n ...",2010-05-08 07:28:36,45 days 23:59:33,4.766667
